In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.04', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54
1,IIT: Pct Change due to behavior,-2.84,-2.35,-1.63,-1.01,-0.39,0.25,1.00,1.92,3.13,4.77,0.28,8.03
2,IIT: Pct Change due to macro,5.33,10.69,16.42,22.55,29.15,36.19,43.71,51.71,60.19,69.17,34.94,62.91
3,IIT: Overall Pct Change in taxes,-6.40,-1.14,4.75,10.95,17.66,24.87,32.76,41.42,51.10,62.10,23.76,60.97
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,15.92,34.86,56.03,78.91,103.75,130.58,159.71,191.40,226.21,264.92,126.21,254.07
6,CIT: Pct Change due to macro,-11.57,-18.65,-24.64,-29.70,-34.01,-37.71,-40.91,-43.69,-46.13,-48.26,-37.42,-50.13
7,CIT: Overall Pct Change in taxes,2.51,9.71,17.58,25.78,34.46,43.63,53.46,64.08,75.74,88.80,41.57,76.58
8,All: Pct Change due to tax rates,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.05
9,All: Pct Change due to behavior,-1.70,-0.09,1.87,3.84,5.94,8.17,10.66,13.46,16.72,20.62,7.93,23.26


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60,-5.06
9,Rev Change Due to Behavior,-0.09,-0.00,0.11,0.23,0.36,0.52,0.71,0.93,1.20,1.54,5.50
10,Rev Change Due to Macro,0.21,0.45,0.73,1.02,1.34,1.69,2.11,2.54,3.02,3.54,16.63
11,Total Revenue Change,-0.30,-0.03,0.31,0.70,1.14,1.65,2.26,2.95,3.77,4.74,17.19


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.40,-0.43,-0.44,-0.46,-0.48,-0.49,-0.51,-0.53,-0.55,-4.66
1,Rev Change Due to Behavior,-0.12,-0.11,-0.08,-0.05,-0.02,0.01,0.06,0.12,0.20,0.31,0.31
2,Rev Change Due to Macro,0.23,0.50,0.82,1.17,1.56,2.02,2.53,3.10,3.75,4.48,20.15
3,Total Revenue Change,-0.27,-0.05,0.24,0.57,0.95,1.39,1.89,2.48,3.18,4.02,14.39


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58,-4.81
1,Rev Change Due to Behavior,-0.12,-0.11,-0.08,-0.05,-0.02,0.01,0.06,0.12,0.20,0.32,0.32
2,Rev Change Due to Macro,0.23,0.52,0.84,1.20,1.61,2.08,2.62,3.22,3.90,4.66,20.88
3,Total Revenue Change,-0.28,-0.06,0.24,0.58,0.98,1.43,1.96,2.58,3.31,4.18,14.93


In [8]:
# jason's get-around (with weifeng's correction in line 6)

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * (tc_reform + df_levels.loc[1, df_levels.columns[1:]])
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.32,-0.33,-0.34,-0.35,-0.36,-0.37,-0.38,-0.39,-0.41,-3.27
1,Rev Change Due to Behavior,-0.12,-0.11,-0.08,-0.05,-0.02,0.01,0.06,0.11,0.19,0.30,0.30
2,Rev Change Due to Macro,0.23,0.48,0.77,1.11,1.50,1.96,2.48,3.08,3.78,4.59,19.96
3,Total Revenue Change,0.10,0.05,0.36,0.71,1.13,1.61,2.16,2.81,3.58,4.49,16.99


In [9]:
df_levels.to_csv('og_usa_result_w_tcja_prod_1.04.csv')